# CineMatch: modelos de similaridade item-item

Compara, no **mesmo conjunto de itens**:

1. **Sentence Transformers + cosseno** (embeddings `emb_v1` / `emb_v2` do pgvector; opcionalmente outro modelo), com **TF-IDF + cosseno** como baseline e **RRF** (fusão de ranks).
2. **Rocchio**: (a) *pseudo-relevance feedback* item → item; (b) **perfil** a partir de vários itens curtidos/não curtidos.
3. **Rerankers supervisionados**: **LogisticRegression**, **SGDClassifier** e **LightGBM (LambdaRank)** sobre candidatos TF-IDF ∪ embedding.

> **Escopo (spec §8, ADR-017):** não há interações de usuários. Os modelos supervisionados aprendem o **rótulo fraco** "mesma coleção TMDB" (franquia), com validação **GroupKFold por coleção**. Todas as métricas são **proxies**; não medem relevância para pessoas e não são NDCG de usuários.

**Pré-requisitos**

```bash
docker-compose up -d postgres mlflow        # MLflow do CineMatch em http://127.0.0.1:5010
uv sync --extra ml --group notebook
uv run cinematch features tfidf             # se ainda não existir
uv run jupyter lab models/
```

O código fica em `src/cinematch/similarity/` (testado); este notebook só orquestra e mostra. A mesma comparação roda sem notebook: `uv run cinematch evaluate similarity-models --mlflow`.

In [ ]:
import os
import time
from pathlib import Path

# .env e caminhos relativos (data/...) são resolvidos a partir da raiz do repositório.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)

os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cinematch.config import get_settings
from cinematch.db import get_engine
from cinematch.features.tfidf import load_latest_tfidf
from cinematch.similarity import methods as m
from cinematch.similarity import rerank as r
from cinematch.similarity.data import load_similarity_data, stored_model_versions

# ---- Parâmetros do experimento ----
K = 10  # tamanho da lista avaliada
DENSE = "emb_v2"  # embedding usado em RRF, Rocchio e geração de candidatos
DEPTH = 50  # candidatos por representação no reranker
N_SPLITS = 5  # GroupKFold por coleção
RUN_EXTRA_ST = False  # True: baixa e testa all-MiniLM-L6-v2 (~90 MB, ~1-2 min em CPU)
LOG_MLFLOW = False  # True: registra todos os runs no MLflow do CineMatch
EXPERIMENT = "similarity-item2item-notebook"

pd.set_option("display.max_colwidth", 60)
settings = get_settings()
engine = get_engine()
print("MLflow:", settings.mlflow_tracking_uri)

## Dados
TF-IDF mais recente + todos os embeddings gravados, alinhados por `item_id`. Coleções TMDB vêm de `catalog_items.collection_id`.

In [ ]:
t0 = time.perf_counter()
versions = stored_model_versions(engine)
data = load_similarity_data(engine, versions, load_latest_tfidf(engine))
print(f"{data.n} itens | fingerprint {data.fingerprint} | {time.perf_counter() - t0:.1f}s")
print("filmes/séries:", dict(zip(*np.unique(data.item_type, return_counts=True))))
print("consultas com coleção (>= 2 membros):", data.collection_queries().size)
pd.Series(data.versions, name="versão").to_frame()

## 1. Sentence Transformers + cosseno (baseline TF-IDF)

`neighbor_metrics` reporta, para o top-10 de cada item:

| Métrica | Lê-se como |
|---|---|
| `collection_recall@10` / `collection_ndcg@10` | acha outros filmes da mesma franquia (rótulo fraco) |
| `genre_overlap@10` | vizinhos com gênero em comum (coerência temática) |
| `title_token_overlap@10` | vazamento do título (menor = menos atalho lexical) |
| `catalog_coverage@10` | fração do catálogo que aparece em algum top-10 |
| `hub_top1pct_share@10` | fatia das recomendações ocupada pelo 1% de itens mais recomendados (*hubness*) |

In [ ]:
if RUN_EXTRA_ST:
    # Outro modelo Sentence Transformers, só em memória (não grava no pgvector).
    from cinematch.features.embeddings import SentenceTransformerEncoder
    from cinematch.features.text import embedding_text_v1

    enc = SentenceTransformerEncoder("sentence-transformers/all-MiniLM-L6-v2")
    t0 = time.perf_counter()
    vecs = enc.encode([embedding_text_v1(d) for d in data.docs], batch_size=64)
    data.add_dense("minilm_l6_v1", vecs, f"{enc.model_id}|emb_text_v1")
    print(f"all-MiniLM-L6-v2: {vecs.shape} em {time.perf_counter() - t0:.0f}s")

results, neighbors = {}, {}


def evaluate(name, scorer):
    t0 = time.perf_counter()
    nb = m.rank_all(scorer, data.n, K)
    neighbors[name] = nb
    results[name] = {
        **m.neighbor_metrics(nb, data, K),
        "seconds": round(time.perf_counter() - t0, 2),
    }


COLS = [
    "collection_recall@10",
    "collection_ndcg@10",
    "genre_overlap@10",
    "title_token_overlap@10",
    "same_type@10",
    "catalog_coverage@10",
    "hub_top1pct_share@10",
    "seconds",
]


def table(names=None):
    df = pd.DataFrame(results).T[COLS]
    return df.loc[names] if names else df


evaluate("tfidf:cosine", m.cosine_scorer(data, "tfidf"))
for rep in data.dense:
    evaluate(f"{rep}:cosine", m.cosine_scorer(data, rep))
evaluate(
    f"rrf(tfidf+{DENSE})",
    m.rrf_scorer([m.cosine_scorer(data, "tfidf"), m.cosine_scorer(data, DENSE)], data.n),
)
table()

## 2. Rocchio

**2a. PRF (item → item):** `q' = α·q + β·média(top-m vizinhos de q)`; reordena com o vetor expandido. Pequena grade em `β` e `m`.

In [ ]:
grid = []
for rep in ["tfidf", DENSE]:
    for beta in (0.5, 0.75):
        for mm in (3, 5, 10):
            nb = m.rank_all(m.rocchio_prf_scorer(data, rep, beta=beta, m=mm), data.n, K)
            met = m.neighbor_metrics(nb, data, K)
            grid.append({"rep": rep, "beta": beta, "m": mm, **{c: met[c] for c in COLS[:-1]}})
grid = pd.DataFrame(grid)
grid

In [ ]:
# Melhor configuração por representação, pelo recall de coleção (empate: gênero).
for rep, g in grid.groupby("rep"):
    best = g.sort_values(["collection_recall@10", "genre_overlap@10"], ascending=False).iloc[0]
    evaluate(
        f"{rep}:rocchio_prf(b={best.beta},m={int(best.m)})",
        m.rocchio_prf_scorer(data, rep, beta=float(best.beta), m=int(best.m)),
    )
table()

**2b. Perfil (vários itens):** simula "usuário curtiu 2 filmes de uma franquia". Para cada coleção com ≥ 3 membros, 2 viram o perfil e o restante é o alvo. Compara: só a 1ª seed, **MaxSim** (máximo por seed) e **Rocchio** (média dos vetores).

In [ ]:
tasks = m.multiseed_tasks(data, n_seeds=2, seed=0)
multiseed = {}
for rep in ["tfidf", DENSE]:
    multiseed.update(m.evaluate_multiseed(m.profile_strategies(data, rep), tasks, K))
pd.DataFrame(multiseed).T

**Demo do perfil:** escolha títulos curtidos e (opcional) não curtidos. O termo `γ` empurra para longe dos não curtidos.

In [ ]:
def find(title):
    exact = [h for h in data.search(title, 50) if h[3].lower() == title.lower()]
    hits = exact or data.search(title, 1)
    if not hits:
        raise KeyError(title)
    return hits[0][1]


LIKED = ["Toy Story 3", "Finding Nemo"]
DISLIKED = ["Cars"]

liked = [find(t) for t in LIKED]
disliked = [find(t) for t in DISLIKED]
print(
    "curtidos:",
    [data.docs[data.index_of(i)].title for i in liked],
    "| não curtidos:",
    [data.docs[data.index_of(i)].title for i in disliked],
)
pd.DataFrame(
    {
        f"{rep} | só curtidos": [t for _, t, _ in m.recommend_for_profile(data, rep, liked, k=K)]
        for rep in ["tfidf", DENSE]
    }
    | {
        f"{rep} | γ=0.5 com não curtidos": [
            t for _, t, _ in m.recommend_for_profile(data, rep, liked, disliked, k=K, gamma=0.5)
        ]
        for rep in ["tfidf", DENSE]
    }
)

## 3. Rerankers supervisionados (LR, SGD, LightGBM)

1. Candidatos = top-50 TF-IDF ∪ top-50 `DENSE` por item.
2. Features por par: cossenos de todas as representações, ranks, Jaccard de gêneros/keywords, diretores e elenco em comum, mesmo tipo, mesmo idioma, distância de ano, votos/popularidade do candidato. **Sem features de título.**
3. Rótulo fraco: mesma coleção TMDB. GroupKFold por coleção → predição *out-of-fold*.

In [ ]:
t0 = time.perf_counter()
cands = r.build_candidates(data, DENSE, depth=DEPTH)
print(f"{time.perf_counter() - t0:.1f}s |", cands.meta)
print(f"taxa de positivos: {cands.y.mean():.4%}")
pd.DataFrame(cands.X, columns=cands.features).describe().T[["mean", "std", "min", "max"]]

In [ ]:
details = {}
for name in r.SUPERVISED:
    res = r.cross_validate_reranker(name, cands, data, k=K, n_splits=N_SPLITS)
    key = f"rerank:{name}"
    neighbors[key] = res.neighbors
    results[key] = {**m.neighbor_metrics(res.neighbors, data, K), "seconds": res.fit_seconds}
    details[key] = {
        "importance": res.importance,
        "fold_recall": res.fold_scores,
        "candidates": cands.meta,
        "model": res.model,
    }
    print(f"{key:18s} recall por fold: {res.fold_scores}")
table([k for k in results if k.startswith("rerank:")])

**Ablação:** LightGBM só com cossenos e ranks (sem metadados). Mostra quanto do ganho vem de tipo, idioma, elenco etc.

In [ ]:
keep = [i for i, f in enumerate(cands.features) if f.startswith(("cos_", "rank_"))]
abl = r.Candidates(
    cands.query,
    cands.cand,
    cands.X[:, keep],
    [cands.features[i] for i in keep],
    cands.y,
    cands.meta,
)
res = r.cross_validate_reranker("lightgbm", abl, data, k=K, n_splits=N_SPLITS)
key = "rerank:lightgbm_sem_metadados"
neighbors[key] = res.neighbors
results[key] = {**m.neighbor_metrics(res.neighbors, data, K), "seconds": res.fit_seconds}
details[key] = {
    "importance": res.importance,
    "fold_recall": res.fold_scores,
    "candidates": cands.meta,
    "model": res.model,
}
table([k for k in results if k.startswith("rerank:")])

In [ ]:
# Importância: LR/SGD = coeficientes padronizados (com sinal); LightGBM = ganho normalizado.
BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e"
fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
for ax, name in zip(axes, r.SUPERVISED):
    imp = pd.Series(details[f"rerank:{name}"]["importance"]).sort_values()
    ax.barh(imp.index, imp.values, color=BLUE, height=0.6)
    ax.axvline(0, color=MUTED, linewidth=0.8)
    ax.set_title(name, color=INK, loc="left")
    ax.grid(axis="x", color="#e5e4e0", linewidth=0.6)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(colors=MUTED, labelsize=9)
fig.suptitle("Importância das features no reranker", x=0.01, ha="left", color=INK)
plt.show()

## 4. Comparação geral
Eixo Y: proxy de franquia (rótulo usado no treino dos rerankers). Eixo X: coerência de gênero (não usada no treino).

In [ ]:
summary = table().sort_values("collection_recall@10", ascending=False)
display(
    summary.style.format("{:.3f}", subset=COLS[:-1]).background_gradient(
        subset=["collection_recall@10", "genre_overlap@10"], cmap="Blues"
    )
)

fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)
for fam, color, marker in [
    ("não supervisionado", BLUE, "o"),
    ("reranker supervisionado", ORANGE, "s"),
]:
    sel = summary[
        [(n.startswith("rerank:")) == (fam != "não supervisionado") for n in summary.index]
    ]
    ax.scatter(
        sel["genre_overlap@10"],
        sel["collection_recall@10"],
        s=70,
        color=color,
        marker=marker,
        label=fam,
        edgecolor="white",
        linewidth=1.5,
        zorder=3,
    )
# Rótulos diretos; pontos próximos têm o rótulo empilhado para não colidir.
placed = []
xmax = summary["genre_overlap@10"].max()
for name, row in summary.sort_values("collection_recall@10", ascending=False).iterrows():
    x, y = row["genre_overlap@10"], row["collection_recall@10"]
    near = sum(abs(x - px) < 0.03 and abs(y - py) < 0.03 for px, py in placed)
    placed.append((x, y))
    right_edge = x > xmax - 0.03
    ax.annotate(
        name,
        (x, y),
        xytext=(-8 if right_edge else 6, 4 - 12 * near),
        textcoords="offset points",
        fontsize=8,
        color=MUTED,
        ha="right" if right_edge else "left",
    )
ax.set_xlabel("genre_overlap@10 (coerência temática)", color=MUTED)
ax.set_ylabel("collection_recall@10 (proxy de franquia)", color=MUTED)
ax.grid(color="#e5e4e0", linewidth=0.6)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="lower left")
ax.set_title("Trade-off: franquia × tema (k=10)", loc="left", color=INK)
plt.show()

### Revisão qualitativa
Top-5 de cada método para alguns títulos. Troque a lista à vontade.

In [ ]:
QUERIES = ["The Dark Knight", "Toy Story", "Interstellar", "Breaking Bad", "Pride & Prejudice"]
METHODS = [
    "tfidf:cosine",
    f"{DENSE}:cosine",
    f"rrf(tfidf+{DENSE})",
    "rerank:logreg",
    "rerank:lightgbm",
]

for q in QUERIES:
    try:
        idx = data.index_of(find(q))
    except KeyError:
        print(f"'{q}' fora do catálogo")
        continue
    print(f"\n=== {data.docs[idx].title} ({data.docs[idx].item_type}) ===")
    display(
        pd.DataFrame(
            {meth: [data.docs[j].title for j in neighbors[meth][idx, :5]] for meth in METHODS}
        )
    )

## 5. MLflow
Com `LOG_MLFLOW = True`, cada método vira um run com proveniência (fingerprint dos dados, versões, commit) e a tag `label_kind=weak:tmdb_collection`. Rerankers levam modelo (`model/model.joblib`) e importâncias.

In [ ]:
if LOG_MLFLOW:
    from cinematch.similarity.benchmark import log_benchmark

    out = {
        "k": K,
        "fingerprint": data.fingerprint,
        "versions": data.versions,
        "item_to_item": results,
        "multiseed": multiseed,
    }
    log_benchmark(out, details, EXPERIMENT, DENSE, DEPTH, N_SPLITS)
    print(f"runs em {settings.mlflow_tracking_uri} (experimento '{EXPERIMENT}')")
else:
    print("LOG_MLFLOW=False: nada registrado")

## Leitura e cuidados

- **Circularidade:** os rerankers são treinados e avaliados no mesmo tipo de rótulo (coleção), ainda que em coleções diferentes (GroupKFold). Ganho em `collection_recall` era esperado; o sinal mais informativo é o que acontece nas métricas **não** treinadas (gênero, título, cobertura, hubness).
- **Viés do rótulo:** coleções só existem para filmes. O reranker aprende "mesmo tipo" com peso alto e aplica isso às séries, que não têm rótulo. Ver `same_type@10`.
- **Franquia ≠ preferência:** um sistema que só devolve sequências é pouco útil. A escolha final do `/similar` depende de dados de uso (Fase 4/5: CTR, favoritos, com atribuição de exposição).
- Resultados medidos e decisão: `docs/experiments/similarity_models_2026-11-10.md` e ADR-017.